In [ ]:
python
class DecoderRNN(nn.Module):
　　def __init__(self, hidden_size, output_size):
　　　　super(DecoderRNN, self).__init__()
　　　　self.embedding=nn.Embedding(output_size, hidden_size)
　　　　self.rnn=nn.RNN(hidden_size, hidden_size, batch_first=True)
　　　　self.out=nn.Linear(hidden_size, output_size)
　　def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
　　　　batch_size=encoder_outputs.size(0)
　　　　decoder_input=th.empty(batch_size, 1, dtype=th.long).fill_(SOS_token)　　# Start of Sentence词元，表示开始生成一个句子
　　　　decoder_hidden=encoder_hidden
　　　　decoder_outputs=[]
　　　　for i in range(MAX_LENGTH):
　　　　　　decoder_output, decoder_hidden　=self.forward_step(decoder_input, decoder_hidden)
　　　　　　decoder_outputs.append(decoder_output)
　　　　　　if target_tensor is not None:　　# 强制学习
　　　　　　　　decoder_input=target_tensor[:, i].unsqueeze(1)
　　　　　　else:
　　　　　　　　_, topi=decoder_output.topk(1)
　　　　　　decoder_input=topi.squeeze(-1).detach()
　　　　decoder_outputs=th.cat(decoder_outputs, dim=1)
　　　　decoder_outputs=F.log_softmax(decoder_outputs, dim=-1)
　　　　return decoder_outputs, decoder_hidden, None
　　def forward_step(self, x, hidden):
　　　　x=self.embedding(x)
　　　　x=F.relu(x)
　　　　x, hidden=self.rnn(x, hidden)
　　　　output=self.out(x)
　　　　return output, hidden

In [ ]:
python
n_epochs=100
batch_size=32
MAX_LENGTH=11
hidden_size=128
learning_rate=0.001
dataloader=DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
encoder=EncoderRNN(dataset.num_word, hidden_size)
decoder=AttentionDecoderRNN(hidden_size, dataset.num_word)
encoder_optimizer=optim.Adam(encoder.parameters(), lr=learning_rate)
decoder_optimizer=optim.Adam(decoder.parameters(), lr=learning_rate)
criterion=nn.NLLLoss()
for i in range(n_epochs+1):
　　total_loss=0
　　for input_tensor, target_tensor, target_length in dataloader:
　　　　encoder_optimizer.zero_grad()
　　　　decoder_optimizer.zero_grad()
　　　　encoder_outputs, encoder_hidden=encoder(input_tensor)
　　　　decoder_outputs, _, _=decoder(encoder_outputs, encoder_hidden, target_tensor)
　　　　loss=criterion(
　　　　　　decoder_outputs.view(-1, decoder_outputs.size(-1)),
　　　　　　target_tensor.view(-1).long()
　　　　)
　　　　loss.backward()
　　　　encoder_optimizer.step()
　　　　decoder_optimizer.step()
　　　　total_loss+=loss.item()
　　total_loss/=len(dataloader)
　　if　i % 10==0:
　　　　print(f"epoch: {i}, loss: {total_loss}")